# NB-07: 2026年全レース めぐ指数計算 → STG DB保存

**目的**: NB-01/NB-02で確立したモデルを使い、2026年全レースのめぐ指数を計算してSTG DBに保存する。

## 処理フロー
```
page_reference/tables/2026/race_result_flat.parquet
  ↓ 前半スプリット・TSI・斤量補正計算
  ↓ megu_regression_params (STG DB) から β₁〜β₄ 取得
  ↓ megu_par_time (STG DB) から par_time 取得
  ↓ めぐ指数計算: 100 + (par_time - adjusted_time) × 10
  ↓ megu_index テーブルに UPSERT
```

## prod 環境への適用
`src/pipeline/megu_index/compute.py` のコード（notebook のエクスポート版）が
prod 環境で同一ロジックを実行する。DB接続先は `KEIBA_ENV=prod` で切替。


In [1]:
import sys, json, warnings, os
sys.path.insert(0, '/home/jovyan/work/keiba-vpn')
warnings.filterwarnings('ignore')
os.environ.setdefault('KEIBA_ENV', 'stg')

import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from sqlalchemy import text
from src.db.session import get_session, init_engine

init_engine()

PROJECT_ROOT = Path('/home/jovyan/work/keiba-vpn')
TABLES_DIR   = PROJECT_ROOT / 'data/page_reference/tables'
CUSHION_DIR  = PROJECT_ROOT / 'data/page_reference/cushion'
NB01_OUTPUT  = PROJECT_ROOT / 'notebooks/megu_index/output/nb01'
MODEL_VERSION = 'v1'

print('Setup OK  (KEIBA_ENV=%s)' % os.environ.get('KEIBA_ENV'))


Setup OK  (KEIBA_ENV=stg)


## 1. 回帰パラメータをSTG DBから取得

In [2]:
with get_session() as s:
    rows = s.execute(text(
        'SELECT param_name, param_value FROM megu_regression_params WHERE model_version=:mv'
    ), {'mv': MODEL_VERSION}).fetchall()

params = {r[0]: float(r[1]) for r in rows}
print('回帰パラメータ:')
for k, v in params.items():
    print(f'  {k}: {v:.6f}')

BETA_PACE   = params.get('beta_pace', 0.0)
BETA_TRACK  = params.get('beta_track', 0.0)
BETA_WEIGHT = params.get('beta_weight', 0.0)
BETA_LEVEL  = params.get('beta_level', 0.0)
TSI_MEAN    = params.get('tsi_mean', 9.17)

print(f'\nβ₁ (pace):   {BETA_PACE:.6f}')
print(f'β₂ (track):  {BETA_TRACK:.6f}')
print(f'β₃ (weight): {BETA_WEIGHT:.6f}')
print(f'β₄ (level):  {BETA_LEVEL:.6f}')
print(f'TSI_mean:    {TSI_MEAN:.4f}')


回帰パラメータ:
  beta_pace: 0.837654
  beta_track: 0.185492
  beta_weight: 0.568264
  beta_level: 0.000000
  tsi_mean: 2.736125
  r2_adj: 0.940770

β₁ (pace):   0.837654
β₂ (track):  0.185492
β₃ (weight): 0.568264
β₄ (level):  0.000000
TSI_mean:    2.7361


## 2. par_time をSTG DBから取得

In [3]:
with get_session() as s:
    par_rows = s.execute(text('''
        SELECT distance, course, surface, track_condition,
               par_time_sec, par_front_split_sec
        FROM megu_par_time WHERE model_version=:mv
    '''), {'mv': MODEL_VERSION}).fetchall()

df_par = pd.DataFrame(par_rows, columns=['distance','course','surface','track_condition',
                                          'par_time_sec','par_front_split_sec'])
df_par['par_time_sec'] = pd.to_numeric(df_par['par_time_sec'], errors='coerce')
df_par['par_front_split_sec'] = pd.to_numeric(df_par['par_front_split_sec'], errors='coerce')
df_par['distance'] = pd.to_numeric(df_par['distance'], errors='coerce').astype(int)
print(f'par_time セル数: {len(df_par):,}')
print(df_par[df_par['surface']=='芝'].sort_values('par_time_sec').head(10).to_string())


par_time セル数: 174
    distance course surface track_condition  par_time_sec  par_front_split_sec
6       1000      直       芝               良         56.41                22.13
5       1000      直       芝              稍重         57.36                22.52
4       1000      右       芝               良         57.66                22.13
7       1000      直       芝            重・不良         58.24                22.52
3       1000      右       芝              稍重         58.31                22.52
21      1200      左       芝               良         70.07                33.94
18      1200      右       芝               良         70.13                33.94
20      1200      左       芝              稍重         70.68                34.09
17      1200      右       芝              稍重         70.85                34.09
22      1200      左       芝            重・不良         71.09                34.33


## 3. 2026年データの前処理

In [4]:

# NB-01と同じロジックを再定義
def parse_lap_times(lap_json, distance):
    if pd.isna(lap_json) or not lap_json:
        return {}
    try:
        laps = json.loads(lap_json) if isinstance(lap_json, str) else lap_json
        if not laps:
            return {}
        seg_len = distance / len(laps)
        result = {}
        cumtime = 0.0
        cumdist = 0.0
        for t in laps:
            cumtime += t
            cumdist += seg_len
            result[round(cumdist)] = round(cumtime, 2)
        return result
    except:
        return {}

def select_split_point(distance, available_dists):
    target = distance * 0.5
    cands = [d for d in sorted(available_dists) if d <= target]
    return max(cands) if cands else None

# 2026年データロード
df_2026 = pd.read_parquet(TABLES_DIR / '2026/race_result_flat.parquet')
df_2026['finish_time_sec'] = pd.to_numeric(df_2026['time_sec'], errors='coerce')
df_2026['finish_pos'] = pd.to_numeric(df_2026['finish_position'], errors='coerce')

# STG DBのracesテーブルから surface/distance を補完（flat_fileに欠損あり）
with get_session() as s:
    df_races_db = pd.read_sql(text(
        'SELECT race_id, surface, distance, direction FROM races WHERE is_excluded=false'
    ), s.bind)

df_2026 = df_2026.merge(
    df_races_db.rename(columns={'surface': 'surface_db', 'distance': 'distance_db', 'direction': 'direction_db'}),
    on='race_id', how='left'
)
df_2026['surface'] = df_2026['surface'].fillna(df_2026['surface_db'])
df_2026['distance'] = pd.to_numeric(df_2026['distance'], errors='coerce').fillna(
    pd.to_numeric(df_2026['distance_db'], errors='coerce'))
df_2026['direction'] = df_2026.get('direction', pd.Series(dtype=str)).fillna(df_2026['direction_db'])

# 平地（芝・ダート）のみ
df_2026 = df_2026[df_2026['surface'].isin(['芝', 'ダート'])].copy()
df_2026 = df_2026[df_2026['finish_time_sec'].notna()]
df_2026 = df_2026[df_2026['distance'] > 0]
print(f'2026年データ: {len(df_2026):,} 行  レース数: {df_2026["race_id"].nunique():,}')
print(df_2026['surface'].value_counts().to_dict())


2026年データ: 6,919 行  レース数: 486
{'芝': 6642, 'ダート': 277}


In [5]:
# ── 前半スプリット計算 ─────────────────────────────────────────────────
splits_2026 = []
for _, row in df_2026[['race_id','distance','lap_times']].drop_duplicates('race_id').iterrows():
    lap_dict = parse_lap_times(row['lap_times'], row['distance'])
    if not lap_dict:
        splits_2026.append({'race_id': row['race_id'], 'front_split_sec': np.nan, 'split_point_m': np.nan})
        continue
    sp = select_split_point(int(row['distance']), list(lap_dict.keys()))
    if sp is None:
        splits_2026.append({'race_id': row['race_id'], 'front_split_sec': np.nan, 'split_point_m': np.nan})
    else:
        splits_2026.append({'race_id': row['race_id'], 'front_split_sec': lap_dict[sp], 'split_point_m': int(sp)})

df_splits_2026 = pd.DataFrame(splits_2026)
df_2026 = df_2026.merge(df_splits_2026, on='race_id', how='left')
print(f'front_split_sec カバレッジ: {df_2026["front_split_sec"].notna().mean():.1%}')

# ── cushion / TSI_raw ───────────────────────────────────────────────────
import json as _json
cushion_data = _json.loads((CUSHION_DIR / 'cushion_2026.json').read_text())
df_c = pd.DataFrame(cushion_data)
df_c = df_c[df_c['is_race_day']==True].copy()
df_c['date'] = pd.to_datetime(df_c['date'], errors='coerce')
df_c = df_c.dropna(subset=['date'])
df_c['date_str'] = df_c['date'].dt.strftime('%Y-%m-%d')
df_c['venue_code'] = df_c['venue_code'].astype(str).str.strip()

df_2026['date_str'] = pd.to_datetime(df_2026['date'], errors='coerce').dt.strftime('%Y-%m-%d')
df_2026['venue_code'] = df_2026['venue_code'].astype(str).str.strip()
df_c_merge = df_c[['date_str','venue_code','cushion_value','dirt_moisture_goal']].drop_duplicates(subset=['date_str','venue_code'])
df_2026 = df_2026.merge(df_c_merge, on=['date_str','venue_code'], how='left')

df_2026['tsi_raw'] = np.where(
    df_2026['surface'] == '芝',
    df_2026['cushion_value'].fillna(TSI_MEAN),
    -df_2026['dirt_moisture_goal'].fillna(-TSI_MEAN)
)
df_2026['tsi_normalized'] = df_2026['tsi_raw'] - TSI_MEAN

# ── 斤量偏差 ─────────────────────────────────────────────────────────
df_2026['sex'] = df_2026['sex_age'].str.extract(r'^(牡|牝|セン)', expand=False).fillna('牡')
df_2026['base_weight'] = np.where(df_2026['sex']=='牝', 53.0, 55.0)
df_2026['jockey_weight_num'] = pd.to_numeric(df_2026['jockey_weight'], errors='coerce')
df_2026['weight_dev'] = df_2026['jockey_weight_num'] - df_2026['base_weight']
df_2026['dist_scale'] = df_2026['distance'] / 2000.0

# ── track_cat ────────────────────────────────────────────────────────
track_map = {'良': '良', '稍重': '稍重', '重': '重・不良', '不良': '重・不良'}
df_2026['track_cat'] = df_2026['track_condition'].map(track_map).fillna('良')

print(f'前処理完了: {len(df_2026):,} 行')
print(f'tsi_raw カバレッジ: {df_2026["tsi_raw"].notna().mean():.1%}')
print(f'weight_dev カバレッジ: {df_2026["weight_dev"].notna().mean():.1%}')


front_split_sec カバレッジ: 92.7%
前処理完了: 6,919 行
tsi_raw カバレッジ: 100.0%
weight_dev カバレッジ: 100.0%


## 4. めぐ指数の計算

In [6]:
# ── par_front_split マージ（direction重複を防ぐためグループ集計）─────────
df_par_split = (
    df_par[['distance','surface','track_condition','par_front_split_sec']]
    .rename(columns={'track_condition': 'track_cat'})
    .groupby(['distance','surface','track_cat'])['par_front_split_sec']
    .mean().reset_index()
)
df_2026 = df_2026.merge(df_par_split, on=['distance','surface','track_cat'], how='left')
print(f'after par_split merge: {len(df_2026):,} (重複なし確認)')
df_2026['front_split_dev'] = df_2026['front_split_sec'] - df_2026['par_front_split_sec']

# ── par_time マージ ─────────────────────────────────────────────────────
df_par_key = df_par.rename(columns={'course': 'direction', 'track_condition': 'track_cat'})
df_2026 = df_2026.merge(
    df_par_key[['distance','surface','direction','track_cat','par_time_sec']],
    on=['distance','surface','direction','track_cat'], how='left'
)

# フォールバック: 距離×surface×track_cat（方向なし）→ 方向一致のみにしたい場合のバックアップ
par_fb1 = df_par_key.groupby(['distance','surface','track_cat'])['par_time_sec'].mean().reset_index().rename(
    columns={'par_time_sec': 'par_time_fb1'}
)
df_2026 = df_2026.merge(par_fb1, on=['distance','surface','track_cat'], how='left')
df_2026['par_time_final'] = df_2026['par_time_sec'].fillna(df_2026['par_time_fb1'])

print(f'par_time カバレッジ: {df_2026["par_time_sec"].notna().mean():.1%}')
print(f'par_time_final カバレッジ: {df_2026["par_time_final"].notna().mean():.1%}')

# ── 各補正値 ───────────────────────────────────────────────────────────
df_2026['delta_pace_sec']   = BETA_PACE * df_2026['front_split_dev'].fillna(0)
df_2026['delta_track_sec']  = -BETA_TRACK * df_2026['tsi_normalized'].fillna(0)
df_2026['delta_weight_sec'] = BETA_WEIGHT * df_2026['weight_dev'].fillna(0) * df_2026['dist_scale'].fillna(1)
df_2026['delta_level_sec']  = 0.0  # FQデータ未入力

# ── 補正済み走破タイム & めぐ指数 ─────────────────────────────────────
df_2026['adjusted_time_sec'] = (
    df_2026['finish_time_sec']
    - df_2026['delta_pace_sec']
    - df_2026['delta_track_sec']
    - df_2026['delta_weight_sec']
    - df_2026['delta_level_sec']
)

df_2026['megu_index'] = 100.0 + (df_2026['par_time_final'] - df_2026['adjusted_time_sec']) * 10.0

# 有効レコードのみ
df_valid = df_2026[df_2026['par_time_final'].notna() & df_2026['megu_index'].notna()].copy()

print(f'\n=== めぐ指数計算結果 ===')
print(f'有効レコード: {len(df_valid):,} / {len(df_2026):,}')
print(f'megu_index 統計:')
print(df_valid['megu_index'].describe())
print(f'\n補正値統計:')
for col in ['delta_pace_sec','delta_track_sec','delta_weight_sec']:
    print(f'  {col}: mean={df_valid[col].mean():.4f}  std={df_valid[col].std():.4f}')


after par_split merge: 6,919 (重複なし確認)
par_time カバレッジ: 98.8%
par_time_final カバレッジ: 99.8%

=== めぐ指数計算結果 ===
有効レコード: 6,905 / 6,919
megu_index 統計:
count    6905.000000
mean      112.976479
std       133.818415
min      -296.612364
25%        92.168221
50%       106.370936
75%       119.046952
max      2363.857451
Name: megu_index, dtype: float64

補正値統計:
  delta_pace_sec: mean=-0.2648  std=0.8932
  delta_track_sec: mean=-0.6970  std=0.6969
  delta_weight_sec: mean=1.0750  std=1.0238


## 5. STG DB に UPSERT

In [7]:
from sqlalchemy import text
import math

# バッチサイズで UPSERT
BATCH_SIZE = 500
now = datetime.now()

records = df_valid[['race_id','horse_id',
                     'finish_time_sec','par_time_final',
                     'delta_pace_sec','delta_track_sec','delta_weight_sec','delta_level_sec',
                     'adjusted_time_sec','megu_index',
                     'front_split_sec','split_point_m','tsi_raw']].copy()

total = len(records)
saved = 0

with get_session() as session:
    for start in range(0, total, BATCH_SIZE):
        batch = records.iloc[start:start+BATCH_SIZE]
        for _, row in batch.iterrows():
            def to_val(v):
                if v is None or (isinstance(v, float) and math.isnan(v)):
                    return None
                return float(v)
            session.execute(text('''
                INSERT INTO megu_index
                  (race_id, horse_id, finish_time_sec, par_time_sec,
                   delta_pace_sec, delta_track_sec, delta_weight_sec, delta_level_sec,
                   adjusted_time_sec, megu_index, field_quality,
                   front_split_sec, split_point_m, tsi_raw,
                   model_version, computed_at)
                VALUES
                  (:rid, :hid, :ft, :pt,
                   :dp, :dt, :dw, :dl,
                   :at, :mi, :fq,
                   :fsp, :spm, :tsi,
                   :mv, NOW())
                ON CONFLICT (race_id, horse_id, model_version) DO UPDATE
                SET finish_time_sec=EXCLUDED.finish_time_sec,
                    par_time_sec=EXCLUDED.par_time_sec,
                    delta_pace_sec=EXCLUDED.delta_pace_sec,
                    delta_track_sec=EXCLUDED.delta_track_sec,
                    delta_weight_sec=EXCLUDED.delta_weight_sec,
                    delta_level_sec=EXCLUDED.delta_level_sec,
                    adjusted_time_sec=EXCLUDED.adjusted_time_sec,
                    megu_index=EXCLUDED.megu_index,
                    front_split_sec=EXCLUDED.front_split_sec,
                    split_point_m=EXCLUDED.split_point_m,
                    tsi_raw=EXCLUDED.tsi_raw,
                    computed_at=NOW()
            '''), {
                'rid': str(row['race_id']),
                'hid': str(row['horse_id']),
                'ft':  to_val(row['finish_time_sec']),
                'pt':  to_val(row['par_time_final']),
                'dp':  to_val(row['delta_pace_sec']),
                'dt':  to_val(row['delta_track_sec']),
                'dw':  to_val(row['delta_weight_sec']),
                'dl':  to_val(row['delta_level_sec']),
                'at':  to_val(row['adjusted_time_sec']),
                'mi':  to_val(row['megu_index']),
                'fq':  None,
                'fsp': to_val(row['front_split_sec']),
                'spm': int(row['split_point_m']) if pd.notna(row['split_point_m']) else None,
                'tsi': to_val(row['tsi_raw']),
                'mv':  MODEL_VERSION,
            })
            saved += 1
        session.commit()
        print(f'  {min(start+BATCH_SIZE, total):,}/{total:,} 保存...', end='\r')

print(f'\n✓ 保存完了: {saved:,} 件')


  6,905/6,905 保存...
✓ 保存完了: 6,905 件


## 6. 保存結果の検証

In [8]:
# ── DB確認 ─────────────────────────────────────────────────────────────
with get_session() as s:
    cnt = s.execute(text('SELECT COUNT(*) FROM megu_index WHERE model_version=:mv'), {'mv':MODEL_VERSION}).scalar()
    stats = s.execute(text('''
        SELECT MIN(megu_index), MAX(megu_index), AVG(megu_index),
               AVG(ABS(delta_pace_sec)), AVG(ABS(delta_track_sec)), AVG(ABS(delta_weight_sec))
        FROM megu_index WHERE model_version=:mv
    '''), {'mv':MODEL_VERSION}).fetchone()
    print(f'megu_index テーブル: {cnt:,} 件')
    print(f'  megu_index range: {float(stats[0]):.1f} 〜 {float(stats[1]):.1f}  avg: {float(stats[2]):.2f}')
    print(f'  |Δpace| avg: {float(stats[3]):.4f}秒')
    print(f'  |Δtrack| avg: {float(stats[4]):.4f}秒')
    print(f'  |Δweight| avg: {float(stats[5]):.4f}秒')

# ── サンプルレース確認 ────────────────────────────────────────────────
sample_race = df_valid.iloc[0]['race_id']
with get_session() as s:
    race_info = s.execute(text('SELECT race_name, distance, surface, track_condition FROM races WHERE race_id=:rid'), 
                          {'rid': sample_race}).fetchone()
    results = s.execute(text('''
        SELECT mi.horse_id, rr.finish_pos, mi.finish_time_sec, mi.par_time_sec,
               mi.delta_pace_sec, mi.delta_track_sec, mi.delta_weight_sec,
               mi.adjusted_time_sec, mi.megu_index
        FROM megu_index mi
        JOIN race_results rr ON mi.race_id=rr.race_id AND mi.horse_id=rr.horse_id
        WHERE mi.race_id=:rid AND mi.model_version=:mv
        ORDER BY rr.finish_pos
    '''), {'rid': sample_race, 'mv': MODEL_VERSION}).fetchall()

print(f'\n=== サンプルレース: {sample_race} ===')
if race_info:
    print(f'  {race_info[0]} {race_info[1]}m {race_info[2]} {race_info[3]}')
print(f'  {"着":>3} {"horse_id":>15} {"生タイム":>8} {"par":>8} {"Δpace":>7} {"Δtrack":>7} {"Δweight":>8} {"調整タイム":>9} {"めぐ指数":>9}')
for r in results:
    print(f'  {r[1]:>3} {r[0]:>15} {float(r[2]):>8.2f} {float(r[3]):>8.2f} {float(r[4]):>7.3f} {float(r[5]):>7.3f} {float(r[6]):>8.3f} {float(r[7]):>9.2f} {float(r[8]):>9.1f}')


megu_index テーブル: 6,905 件
  megu_index range: -296.6 〜 2363.9  avg: 112.98
  |Δpace| avg: 0.6880秒
  |Δtrack| avg: 0.7311秒
  |Δweight| avg: 1.1799秒

=== サンプルレース: 202603010102 ===
  3歳未勝利 1200m 芝 良
    着        horse_id     生タイム      par   Δpace  Δtrack  Δweight     調整タイム      めぐ指数
    1      2023106190    68.60    70.13   0.050   0.000    0.000     68.55     115.8
    2      2023105952    68.80    70.13   0.050   0.000    0.682     68.07     120.6
    3      2023107103    69.00    70.13   0.050   0.000    0.682     68.27     118.6
    4      2023105579    69.50    70.13   0.050   0.000   -0.341     69.79     103.4
    5      2023102783    69.50    70.13   0.050   0.000    0.682     68.77     113.6
    6      2023101779    69.50    70.13   0.050   0.000   -0.341     69.79     103.4
    7      2023105833    69.60    70.13   0.050   0.000    0.682     68.87     112.6
    8      2023101675    69.80    70.13   0.050   0.000   -0.341     70.09     100.4
    9      2023101205    70.10    70.13 

## 7. 今週のめぐ指数集計（A/B/Cパターン）の検証

In [9]:
# ── 直近5走のめぐ指数集計 ─────────────────────────────────────────────
# パターンB ウェイト [U-1]: 0.35 / 0.25 / 0.20 / 0.12 / 0.08
WEIGHTS_B = [0.35, 0.25, 0.20, 0.12, 0.08]

# df_validから horse_id × race_dateの順位付き結果を構築
df_agg = df_valid[['race_id','horse_id','date','megu_index','surface','distance']].copy()
df_agg['race_date'] = pd.to_datetime(df_agg['date'])
df_agg = df_agg.sort_values(['horse_id','race_date'])

# 直近5走を取得して集計
results_agg = []
for horse_id, grp in df_agg.groupby('horse_id'):
    grp = grp.sort_values('race_date', ascending=False)
    recent = grp.head(5)
    if len(recent) == 0:
        continue
    
    # パターンA: 最大値
    megu_a = recent['megu_index'].max()
    
    # パターンB: 加重平均
    weights = WEIGHTS_B[:len(recent)]
    weights = np.array(weights) / sum(weights)
    megu_b = (recent['megu_index'].values * weights).sum()
    
    # パターンC: 同距離帯・同馬場での直近3走最大値
    dist_band = grp.iloc[0]['distance']
    if dist_band < 1500:
        dist_label = 'sprint'
    elif dist_band < 1800:
        dist_label = 'mile'
    elif dist_band < 2400:
        dist_label = 'middle'
    else:
        dist_label = 'long'
    surf_c = grp.iloc[0]['surface']
    cond_c = grp[(grp['surface']==surf_c)].head(3)
    megu_c = cond_c['megu_index'].max() if len(cond_c) > 0 else np.nan
    
    results_agg.append({'horse_id': horse_id, 'megu_a': megu_a, 'megu_b': megu_b, 'megu_c': megu_c})

df_horse_megu = pd.DataFrame(results_agg)
print(f'集計対象馬数: {len(df_horse_megu):,}')
print('\nめぐ指数集計統計:')
print(df_horse_megu[['megu_a','megu_b','megu_c']].describe())

# パターン間の相関
corr = df_horse_megu[['megu_a','megu_b','megu_c']].corr()
print('\nパターン間 Spearman 相関:')
print(corr.to_string())


集計対象馬数: 3,864

めぐ指数集計統計:
            megu_a       megu_b       megu_c
count  3864.000000  3864.000000  3864.000000
mean    125.427501   114.044818   124.473040
std     176.191097   133.273594   173.805100
min    -119.873603  -119.873603  -119.873603
25%      95.012171    91.579577    94.812328
50%     110.364067   105.871072   110.296346
75%     124.000956   117.721736   123.900277
max    2363.857451  2258.243643  2363.857451

パターン間 Spearman 相関:
          megu_a    megu_b    megu_c
megu_a  1.000000  0.925117  0.983076
megu_b  0.925117  1.000000  0.928429
megu_c  0.983076  0.928429  1.000000
